# 3.16 Gold — Communication State

Direction-aware communication recency per lead: last inbound vs last outbound **per channel**, plus a `whose_turn` verdict built on the distinction between an **attempt** and **actual contact made**.

## Semantics

**Attempt vs contact (the core distinction):**
- Outbound call, any disposition → **attempt**. Only `disposition = 'answered'` counts as contact.
- Client inbound call, any disposition (incl. no-answer) → **the client reached out** — a missed inbound call still flips the turn to us.
- Sent email / outbound SMS → treated as contact made (delivery assumed; source has no read receipts).
- Slack messages (either direction) → contact made.
- `Meeting` / `Note` / `CustomActivity` carry no direction in Close → `other_days`, informational only, excluded from turn logic.

**Columns:**
- `in_email_days` / `out_email_days`, `in_call_days` / `out_call_days`, `in_sms_days` / `out_sms_days`, `in_slack_days` / `out_slack_days` — days since last in/out per channel (NULL = never)
- `call_connected_days` — last **live phone conversation** (answered call, either direction)
- `missed_client_call_days` — last inbound no-answer (client tried, we missed)
- `other_days` — last Meeting/Note/CustomActivity (no direction in source)
- `client_reach_days` — client's last contact **or attempt** (least of all inbound channels, incl. missed calls)
- `we_connected_days` — our last **contact made** (sent email, sent SMS, answered call, Slack)
- `we_attempted_days` — our last attempt of any kind, **including unanswered calls**
- `whose_turn` — `'US'` if the client reached out more recently than we last made contact (unanswered outbound attempts do NOT clear our turn); `'CLIENT'` if our last contact-made is more recent; `'CURRENT'` if both on the same day
- `all_out_attempted` — call, email, AND SMS each attempted within the window with no client reach in the window ("all outreach made, no response anywhere")

**This table is recognition, not prescription.** It is deliberately NOT joined into the health score or the profile — applying it (contact lists, cadence rules, escalation) is downstream logic the business can layer on top without touching scoring.

In [0]:
# ============================================================================
# CONFIG — business-tunable constants
# ============================================================================
# ATTEMPT_WINDOW_DAYS is used ONLY for the all_out_attempted flag.
# whose_turn uses no window: it compares last dates directly, so it never goes
# stale by threshold choice. The business may retune the window without
# touching any other logic.
ATTEMPT_WINDOW_DAYS = 14

print(f"✅ Config: attempt window = {ATTEMPT_WINDOW_DAYS} days")

In [0]:
# ============================================================================
# BUILD — crm_ingestion.gold.communication_state
# ============================================================================
from pyspark.sql import functions as F

# Sources: gold.leads_activity_fact (Call/Email/SMS/Meeting/Note/CustomActivity,
#          direction + disposition), gold.sentiment (Slack both directions).
# Email uses status (inbox/sent) rather than direction because Close encodes
# email direction as incoming/outgoing + status inbox/sent — two vocabularies,
# one meaning: client-initiated. Normalized here, at the gold layer.
# Population: ALL leads seen in either source (not just profile clients) so the
# business can apply the same recognition to any population later.

spark.sql(f"""
CREATE OR REPLACE TABLE crm_ingestion.gold.communication_state AS

WITH act_dates AS (
    SELECT
        lead_id,
        MAX(CASE WHEN type = 'Email' AND status = 'inbox'
                 THEN to_date(activity_at) END)                          AS in_email_date,
        MAX(CASE WHEN type = 'Email' AND status = 'sent'
                 THEN to_date(activity_at) END)                          AS out_email_date,
        MAX(CASE WHEN type = 'Call' AND direction = 'inbound'
                 THEN to_date(activity_at) END)                          AS in_call_date,
        MAX(CASE WHEN type = 'Call' AND direction = 'outbound'
                 THEN to_date(activity_at) END)                          AS out_call_date,
        MAX(CASE WHEN type = 'Call' AND disposition = 'answered'
                 THEN to_date(activity_at) END)                          AS call_connected_date,
        MAX(CASE WHEN type = 'Call' AND direction = 'inbound'
                        AND disposition = 'no-answer'
                 THEN to_date(activity_at) END)                          AS missed_client_call_date,
        MAX(CASE WHEN type = 'SMS' AND direction = 'inbound'
                 THEN to_date(activity_at) END)                          AS in_sms_date,
        MAX(CASE WHEN type = 'SMS' AND direction = 'outbound'
                 THEN to_date(activity_at) END)                          AS out_sms_date,
        MAX(CASE WHEN type IN ('Meeting', 'Note', 'CustomActivity')
                 THEN to_date(activity_at) END)                          AS other_date
    FROM crm_ingestion.gold.leads_activity_fact
    GROUP BY lead_id
),

slack AS (
    SELECT lead_id,
           days_since_last_message_from_client     AS in_slack_days,
           days_since_last_message_from_team_member AS out_slack_days
    FROM crm_ingestion.gold.sentiment
),

combined AS (
    SELECT
        COALESCE(a.lead_id, s.lead_id) AS lead_id,
        DATEDIFF(current_date, a.in_email_date)         AS in_email_days,
        DATEDIFF(current_date, a.out_email_date)        AS out_email_days,
        DATEDIFF(current_date, a.in_call_date)          AS in_call_days,
        DATEDIFF(current_date, a.out_call_date)         AS out_call_days,
        DATEDIFF(current_date, a.call_connected_date)  AS call_connected_days,
        DATEDIFF(current_date, a.missed_client_call_date) AS missed_client_call_days,
        DATEDIFF(current_date, a.in_sms_date)           AS in_sms_days,
        DATEDIFF(current_date, a.out_sms_date)          AS out_sms_days,
        s.in_slack_days,
        s.out_slack_days,
        DATEDIFF(current_date, a.other_date)            AS other_days
    FROM act_dates a
    FULL OUTER JOIN slack s ON a.lead_id = s.lead_id
)

SELECT
    c.*,

    -- Client's last reach: contact made OR attempt (missed calls count — they tried)
    LEAST(in_email_days, in_call_days, in_sms_days, in_slack_days)
        AS client_reach_days,

    -- Our last contact made: sent email, sent SMS, live answered call (either
    -- direction — a live conversation is a live conversation), Slack message.
    -- Unanswered outbound calls are NOT here — that is the whole point.
    LEAST(out_email_days, out_sms_days, call_connected_days, out_slack_days)
        AS we_connected_days,

    -- Our last attempt of any kind, including unanswered calls (cadence guard:
    -- 'we already tried calling them today')
    LEAST(out_email_days, out_sms_days, out_call_days, out_slack_days)
        AS we_attempted_days

FROM combined c
""")

# whose_turn + all_out_attempted flag need the composites above — add as columns
spark.sql(f"""
CREATE OR REPLACE TABLE crm_ingestion.gold.communication_state AS
SELECT
    *,
    CASE
        WHEN client_reach_days IS NULL AND we_connected_days IS NULL THEN NULL
        WHEN we_connected_days IS NULL
             OR client_reach_days < we_connected_days          THEN 'US'
        WHEN client_reach_days IS NULL
             OR we_connected_days < client_reach_days          THEN 'CLIENT'
        ELSE 'CURRENT'
    END AS whose_turn,

    ( out_call_days  IS NOT NULL AND out_call_days  <= {ATTEMPT_WINDOW_DAYS}
      AND out_email_days IS NOT NULL AND out_email_days <= {ATTEMPT_WINDOW_DAYS}
      AND out_sms_days   IS NOT NULL AND out_sms_days   <= {ATTEMPT_WINDOW_DAYS}
      AND (client_reach_days IS NULL OR client_reach_days > {ATTEMPT_WINDOW_DAYS})
    ) AS all_out_attempted
FROM crm_ingestion.gold.communication_state
""")

result = spark.table("crm_ingestion.gold.communication_state")
print(f"✅ communication_state built: {result.count():,} leads")
result.groupBy("whose_turn").count().orderBy(F.desc("count")).show()

In [0]:
%sql
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT lead_id) AS distinct_leads
FROM crm_ingestion.gold.communication_state;

In [0]:
# ============================================================================
# VALIDATION — how stale is the "waiting on us" backlog?
# Non-client leads whose_turn='US': they made the last move, we never
# connected since. Split by how long they've been waiting, and whether we
# have EVER connected to or attempted them. This is the inbound-interest
# leakage view: people who raised their own hand and got silence.
# ============================================================================
from pyspark.sql import functions as F

_cs = spark.table("crm_ingestion.gold.communication_state")
# customer_health_profile is now SCD2 — open versions (VRSN_END_DATE IS NULL)
# are the current clients; closed versions are departed leads.
_client_ids = (spark.table("crm_ingestion.gold.customer_health_profile")
                 .filter("VRSN_END_DATE IS NULL")
                 .select("lead_id"))
_waiting = (_cs.join(_client_ids, "lead_id", "left_anti")
              .filter(F.col("whose_turn") == "US"))

print(f"Non-client leads currently waiting on us: {_waiting.count():,}")

(_waiting.select(
    F.when(F.col("client_reach_days") <= 3, 0)
     .when(F.col("client_reach_days") <= 7, 1)
     .when(F.col("client_reach_days") <= 14, 2)
     .when(F.col("client_reach_days") <= 30, 3)
     .otherwise(4).alias("ord"),
    F.when(F.col("client_reach_days") <= 3, "0-3 days (live queue)")
     .when(F.col("client_reach_days") <= 7, "4-7 days")
     .when(F.col("client_reach_days") <= 14, "8-14 days")
     .when(F.col("client_reach_days") <= 30, "15-30 days")
     .otherwise("> 30 days (dropped)").alias("waiting_since"),
    F.col("we_connected_days").isNull().alias("never_connected"),
    F.col("we_attempted_days").isNull().alias("never_attempted"),
)
 .groupBy("ord", "waiting_since", "never_connected", "never_attempted")
 .agg(F.count("*").alias("leads"))
 .orderBy("ord", "never_connected", "never_attempted")
 .drop("ord")
 .show(30, truncate=False))